In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.cluster import OPTICS
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from collections import Counter


## Load datasets

In [ ]:
path = '/content/cold_heads_monitoring/Dataset'
arrays = []
data_list = []

for filename in os.listdir(path):
    if filename.endswith('.pkl'):
        with open(os.path.join(path, filename), 'rb') as f:

            arr = np.asarray(pickle.load(f))[:, 1:2, :]
            arrays.append(arr)
            data_list.append(os.path.splitext(filename)[0])



In [ ]:
all_data = arrays
arrays = [np.array(arrays[6]),np.array(arrays[2]),np.array(arrays[4])]


## Balance datasets

In [ ]:
min_len = 2000

balanced_arrays = [arr[:min_len] for arr in arrays]

data = np.concatenate(balanced_arrays, axis=0)

data = np.squeeze(data,1)
data.shape


In [ ]:
labels = np.empty(len(data),dtype=object)
left_index = 0
right_index = min_len

for i in range(len(data_list)):

  labels[left_index:right_index] = i
  left_index += min_len
  right_index += min_len



## Training DBSCAN

## DBSCAN parameters

`eps` defines the neighborhood radius; `min_samples` defines the minimum neighborhood support for a core point. `metric` selects the distance function, and `algorithm` selects the neighbor-search backend.


In [ ]:
pca = PCA(n_components=2)
data = pca.fit_transform(data)


In [ ]:
dbscan = DBSCAN(eps=1.8, min_samples=5)
dbscan.fit(data)
clustering_labels = dbscan.labels_


In [ ]:
def distance(nuovo_campione,metric='chebyshev'):

  cluster_centroids = []
  for label in np.unique(clustering_labels):
      cluster_points = data[clustering_labels == label]
      centroid = np.mean(cluster_points, axis=0)
      cluster_centroids.append(centroid)

  cluster_centroids = np.array(cluster_centroids)

  distanze = pairwise_distances(nuovo_campione.reshape(1, -1), cluster_centroids, metric=metric)

  cluster_più_vicino = np.argmin(distanze)
  return(cluster_più_vicino)

  print(f"Nearest cluster for the new sample: {cluster_più_vicino}")


In [ ]:
n_clusters = len(set(clustering_labels)) - (1 if -1 in clustering_labels else 0)
print("Number of clusters:", n_clusters)

n_anomalies = list(clustering_labels).count(-1)
print("Number of noise points:", n_anomalies)


In [ ]:
df = pd.DataFrame({'X': data[:, 0], 'Y': data[:, 1], 'Cluster': clustering_labels})

sns.scatterplot(data=df, x='X', y='Y', hue='Cluster', palette='viridis', alpha=0.5)
plt.show()


In [ ]:
fig = px.scatter(df, x='X', y='Y', color='Cluster',
                 color_continuous_scale='viridis',
                 title='Cold Heads Clusters',
                 labels={'Cluster': 'Cluster'})

fig.update_xaxes(fixedrange=False)
fig.update_yaxes(fixedrange=False)

fig.show()


In [ ]:
num_indices_correct = np.sum(clustering_labels == labels)

total_indices = len(clustering_labels)
percentage_correct = (num_indices_correct / total_indices) * 100

print("Correctly classified indices:", percentage_correct, "%")


DBSCAN assigns label `-1` to noise points. Their number depends on the distance metric and density parameters; this label alone does not establish a physical anomaly.


When PCA is applied, the first two principal components describe orthogonal directions of decreasing variance. Their interpretation depends on feature scaling and outliers.


## Inference

In [ ]:
new_data= all_data[4][:2000, 0, :]
pca = PCA(n_components=2)
new_data = pca.fit_transform(new_data)


In [ ]:
result = []
for elem in new_data:
  result.append(distance(elem))
count = Counter(result)
elemento_piu_ripetuto = count.most_common(1)[0][0]


In [ ]:
labels = dbscan.fit_predict(new_data)
count = Counter(labels)
elemento_piu_ripetuto = count.most_common(1)[0][0]
